## Project Overview

**Pesa Pulse**, a Nairobi fintech start-up, has just signed a partner who shares its customer directory through an API. Before the marketing team plans regional campaigns, they want to know the basics: how many customers there are, where they are, and what the data even looks like.

You are the junior analyst. Pull the data, explore its nested structure, extract the attributes that matter, and answer "where are our customers?" first with plain Python structures and then with pandas. Finish with a chart marketing can drop straight into a slide.

### Objectives

- Retrieve JSON from an API with requests and handle a failed request
- Navigate nested dictionaries and lists confidently
- Use sets to find unique values and dictionaries to count them
- Write dict comprehensions for aggregation without pandas
- Move the cleaned data into pandas, export it and visualise a distribution

## Task 1 — Retrieve Customer Data Using an API

An API (Application Programming Interface) allows one application to request data from another application.

In this project, we'll use the JSONPlaceholder Users API, which returns 10 sample user records in JSON format.

We need the `requests` library to send the request and retrieve the data.

### Key Concepts

- `requests.get()` sends a request to the API.
- `timeout=10` prevents the program from waiting indefinitely.
- `raise_for_status()` detects HTTP errors.
- `.json()` converts the JSON response into Python objects.
- `try` and `except` handle common failures without displaying a long traceback.

### Code — `get_customers()`

In [ ]:

import requests

def get_customers(url):
    
    try:
        # Send a request to the API and wait for at most 10 seconds.
        response = requests.get(url, timeout=10)

        # Raise an error if the server returns an unsuccessful status.
        response.raise_for_status()

        # Convert the JSON response into Python objects.
        customers = response.json()

        # Confirm that the API returned a list.
        if not isinstance(customers, list):
            print("Error: Expected a list of customers.")
            return []

        return customers

    except requests.exceptions.RequestException as error:
        # Handle connection, timeout and HTTP errors.
        print(f"Failed to retrieve customers: {error}")
        return []


# API endpoint
url = "https://jsonplaceholder.typicode.com/users"

# Retrieve the data
customers = get_customers(url)

# Display the number of records retrieved
print("Total customers:", len(customers))


Total customers: 10


## Task 2 — Explore Nested Dictionaries and Lists

API responses often contain nested data. A customer record might contain an address dictionary, which itself contains a geographic dictionary.


In [2]:
customer = {
    "name": "Leanne Graham",
    "address": {
        "city": "Gwenborough",
        "geo": {
            "lat": "-37.3159",
            "lng": "81.1496"
        }
    }
}

To access the city:

In [3]:
customer["address"]["city"]

'Gwenborough'

To access the latitude:

In [4]:
customer["address"]["geo"]["lat"]

'-37.3159'

Our goal is to explore the structure recursively and display every key and its value's Python type.

### Code — `explore()`

### How It Works

- `isinstance()` checks whether a value is a dictionary or list.
- `.items()` lets us loop through dictionary keys and values.
- `type(value).__name__` returns types such as `str`, `int`, `float`, `dict` and `list`.
- Recursion means a function calls itself to explore deeper levels.

**README Finding:** The deepest requested attributes are latitude and longitude, accessible through `address → geo → lat` and `address → geo → lng`.

In [5]:

def explore(record, indent=0):
    """Print dictionary keys and Python types at each nesting level."""

    # Create spaces to show the nesting level.
    prefix = " " * indent

    # If the record is a dictionary, inspect its keys and values.
    if isinstance(record, dict):
        for key, value in record.items():

            # Print the key and its value's Python type.
            print(f"{prefix}{key}: {type(value).__name__}")

            # Explore nested dictionaries and lists.
            if isinstance(value, (dict, list)):
                explore(value, indent + 2)

    # If the record is a list, inspect each item.
    elif isinstance(record, list):
        for index, value in enumerate(record):
            print(f"{prefix}[{index}]: {type(value).__name__}")

            if isinstance(value, (dict, list)):
                explore(value, indent + 2)


# Explore the first customer.
if customers:
    explore(customers[0])


id: int
name: str
username: str
email: str
address: dict
  street: str
  suite: str
  city: str
  zipcode: str
  geo: dict
    lat: str
    lng: str
phone: str
website: str
company: dict
  name: str
  catchPhrase: str
  bs: str


## Task 3 — Extract Customer Attributes

The original API response is nested, which can make analysis difficult. We will transform it into a flat structure, with one dictionary per customer.

We need these columns:

| Attribute | Location in API |
|---|---|
| id | Top level |
| name | Top level |
| username | Top level |
| email | Top level |
| email_domain | Extracted from email |
| phone | Top level |
| city | `address.city` |
| zipcode | `address.zipcode` |
| latitude | `address.geo.lat` |
| longitude | `address.geo.lng` |
| company | `company.name` |
| website | Top level |

The latitude and longitude values are returned as strings by this API, so we convert them into floating-point numbers using `float()`.

### Code — `extract_attributes()`

### How It Works

- `.get()` retrieves a dictionary value and allows a default if the key is missing.
- `rsplit("@", 1)` splits an email at its final `@` symbol to extract the domain.
- `float()` converts coordinate strings into numeric values.
- `rows.append(row)` adds each flattened record to the results list.

For production-quality data cleaning, malformed coordinate values should also be handled explicitly. The code above assumes any nonempty coordinate can be converted to a float.

In [6]:

def extract_attributes(customers):
    """Flatten nested customer records into simple dictionaries."""

    rows = []

    for customer in customers:

        # Extract nested dictionaries.
        address = customer.get("address", {})
        geo = address.get("geo", {})
        company = customer.get("company", {})

        # Extract the email domain.
        email = customer.get("email", "")
        email_domain = email.rsplit("@", 1)[-1].lower() if "@" in email else ""

        # Convert coordinates into floats.
        latitude = float(geo["lat"]) if geo.get("lat") else float("nan")
        longitude = float(geo["lng"]) if geo.get("lng") else float("nan")

        # Create one flat dictionary per customer.
        row = {
            "id": customer.get("id"),
            "name": customer.get("name"),
            "username": customer.get("username"),
            "email": email,
            "email_domain": email_domain,
            "phone": customer.get("phone"),
            "city": address.get("city"),
            "zipcode": address.get("zipcode"),
            "latitude": latitude,
            "longitude": longitude,
            "company": company.get("name"),
            "website": customer.get("website")
        }

        rows.append(row)

    return rows


# Extract the attributes.
rows = extract_attributes(customers)

# Inspect the first flat record.
if rows:
    print(rows[0])


{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'email_domain': 'april.biz', 'phone': '1-770-736-8031 x56442', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'latitude': -37.3159, 'longitude': 81.1496, 'company': 'Romaguera-Crona', 'website': 'hildegard.org'}


## Task 4 — Use Sets to Find Unique Cities

A set stores unique values, automatically removing duplicates.



In [ ]:
#Suppose the city data is:

["Nairobi", "Mombasa", "Nairobi", "Kisumu"]

['Nairobi', 'Mombasa', 'Nairobi', 'Kisumu']

The unique cities are:

In [8]:
{"Nairobi", "Mombasa", "Kisumu"}

{'Kisumu', 'Mombasa', 'Nairobi'}

This helps marketing understand how many distinct cities appear in the dataset.

### Code — `unique_cities()`

### How to Interpret the Result

If there are 10 customer records and 10 unique cities, every record has a different city in this sample. If there are 10 records and only 4 unique cities, some cities have multiple records.

Marketing should not confuse the number of unique cities with the number of customers.

In [9]:

def unique_cities(customers):
    """Return the unique cities in the customer records."""

    cities = {
        customer["address"]["city"]
        for customer in customers
        if customer.get("address", {}).get("city")
    }

    return cities


# Find unique cities.
cities = unique_cities(customers)

print("Number of customers:", len(customers))
print("Number of unique cities:", len(cities))
print("Unique cities:", sorted(cities))


Number of customers: 10
Number of unique cities: 10
Unique cities: ['Aliyaview', 'Bartholomebury', 'Gwenborough', 'Howemouth', 'Lebsackbury', 'McKenziehaven', 'Roscoeview', 'South Christy', 'South Elvis', 'Wisokyburgh']


## Task 5 — Count Customers by City and Email Domain

### Part A: Count Customers by City Using a Loop

A dictionary stores information as key-value pairs. Here, the city is the key and the number of customers is the value.


In [ ]:
#For example:
{
    "Nairobi": 5,
    "Mombasa": 3,
    "Kisumu": 2
}

{'Nairobi': 5, 'Mombasa': 3, 'Kisumu': 2}

### Code — `city_counts()`

**Important:** `counts.get(city, 0)` returns the current count or zero if the city has not appeared before. Adding one updates the count.

In [11]:

def city_counts(customers):
    """Count the number of customers in each city."""

    counts = {}

    for customer in customers:
        city = customer.get("address", {}).get("city")

        if city:
            # Add one to the existing count, or start at zero.
            counts[city] = counts.get(city, 0) + 1

    return counts


counts = city_counts(customers)

# Sort from most common to least common.
sorted_counts = dict(
    sorted(counts.items(), key=lambda item: item[1], reverse=True)
)

print("Customers by city:")
print(sorted_counts)


Customers by city:
{'Gwenborough': 1, 'Wisokyburgh': 1, 'McKenziehaven': 1, 'South Elvis': 1, 'Roscoeview': 1, 'South Christy': 1, 'Howemouth': 1, 'Aliyaview': 1, 'Bartholomebury': 1, 'Lebsackbury': 1}


### Part B: Count Customers by Email Domain Using a Dictionary Comprehension

- A dictionary comprehension creates a dictionary in a concise way.

- First, extract the email domains and count each domain.

- This approach is appropriate for the small sample in this exercise. For much larger datasets, repeated calls to `.count()` are inefficient; a single-pass dictionary loop would scale better.

In [12]:

def email_domain_counts(customers):
    """Count email domains using a dictionary comprehension."""

    domains = [
        customer["email"].rsplit("@", 1)[-1].lower()
        for customer in customers
        if "@" in customer.get("email", "")
    ]

    # Count each distinct domain using a dictionary comprehension.
    counts = {
        domain: domains.count(domain)
        for domain in set(domains)
    }

    return counts


domain_counts = email_domain_counts(customers)

# Sort from most common to least common.
sorted_domain_counts = dict(
    sorted(
        domain_counts.items(),
        key=lambda item: item[1],
        reverse=True
    )
)

print("Customers by email domain:")
print(sorted_domain_counts)


Customers by email domain:
{'rosamond.me': 1, 'yesenia.net': 1, 'melissa.tv': 1, 'annie.ca': 1, 'dana.io': 1, 'billy.biz': 1, 'kory.org': 1, 'april.biz': 1, 'jasper.info': 1, 'karina.biz': 1}


## Task 6 — Convert the Records into a Pandas DataFrame

- A pandas DataFrame is a two-dimensional table with rows and columns. It is useful for cleaning, analyzing, validating and exporting data.

- We already have a list of flat dictionaries called `rows`. We can convert it directly into a DataFrame.

### Code — `to_dataframe()`

### How to Interpret the Output

- `df.shape` returns (rows, columns). With the standard API response, you should get 10 rows and 12 columns.
- `df.dtypes` identifies whether each column is an integer, string-like object or floating-point number.
- `value_counts()` calculates the frequency of each city.
- The comparison checks whether the pandas counts match the manual counts.

In [13]:

import pandas as pd

def to_dataframe(rows):
    """Convert a list of dictionaries into a DataFrame."""
    return pd.DataFrame(rows)


df = to_dataframe(rows)

# Print the number of rows and columns.
print("DataFrame shape:", df.shape)

# Display column names and data types.
print("\nData types:")
print(df.dtypes)

# Count customers by city using pandas.
pandas_city_counts = df["city"].value_counts().to_dict()

print("\nPandas city counts:")
print(pandas_city_counts)

# Compare pandas results with our Python dictionary.
print("\nDo the counts match?")
print(pandas_city_counts == counts)


DataFrame shape: (10, 12)

Data types:
id                int64
name                str
username            str
email               str
email_domain        str
phone               str
city                str
zipcode             str
latitude        float64
longitude       float64
company             str
website             str
dtype: object

Pandas city counts:
{'Gwenborough': 1, 'Wisokyburgh': 1, 'McKenziehaven': 1, 'South Elvis': 1, 'Roscoeview': 1, 'South Christy': 1, 'Howemouth': 1, 'Aliyaview': 1, 'Bartholomebury': 1, 'Lebsackbury': 1}

Do the counts match?
True


## Task 7 — Export the Results to CSV

- CSV stands for Comma-Separated Values. It is a simple file format that Excel, Power BI, pandas and other analysis tools can read.

We'll create two files:

1. `customers.csv` — all 12 extracted attributes.
2. `city_counts.csv` — each city and its customer count.

### Code — Export the Files

The `index=False` argument prevents pandas from adding an extra index column to the exported CSV.

In [14]:

from pathlib import Path
import pandas as pd

# Create the output folder if it does not exist.
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

# Export customer records without the DataFrame index.
df.to_csv(output_dir / "customers.csv", index=False)

# Convert the city-count dictionary into a DataFrame.
city_counts_df = pd.DataFrame(
    sorted_counts.items(),
    columns=["city", "customer_count"]
)

# Export the city counts without the index.
city_counts_df.to_csv(
    output_dir / "city_counts.csv",
    index=False
)

print("CSV files exported successfully.")


CSV files exported successfully.


## Task 8 — Visualize the Customer Distribution

- A chart helps marketing understand the data more quickly than a printed dictionary.

- We'll use a horizontal bar chart. Because cities may each appear only once, we'll also create an email-domain chart to see whether some domains are more common than others.

### Code — `plot_distribution()`

In [15]:

import matplotlib.pyplot as plt
from pathlib import Path

def plot_distribution(counts, path):
    """Create and save a horizontal bar chart."""

    if not counts:
        print("No data available for the chart.")
        return

    # Sort the categories from most common to least common.
    sorted_items = sorted(
        counts.items(),
        key=lambda item: item[1],
        reverse=True
    )

    labels = [item[0] for item in sorted_items]
    values = [item[1] for item in sorted_items]

    # Create the chart.
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(labels[::-1], values[::-1])

    ax.set_title("Customer Distribution")
    ax.set_xlabel("Number of customers")
    ax.set_ylabel("Category")

    fig.tight_layout()

    # Create the parent folder if needed and save the chart.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Chart saved to: {path}")


# Create both charts.
plot_distribution(
    sorted_counts,
    "output/customer_distribution.png"
)

plot_distribution(
    sorted_domain_counts,
    "output/email_domain_distribution.png"
)


Chart saved to: output\customer_distribution.png
Chart saved to: output\email_domain_distribution.png


### Which Chart Is More Useful?

- The city chart directly addresses the question, “Where are our customers?” 
- However, if every sample record has a different city, it won't reveal any dominant locations. 
- The email-domain chart may show more variation, but email domains are not a substitute for geographic analysis.

For a real marketing campaign, use a larger, representative customer dataset before deciding where to invest.